In [ ]:
import numpy as np
import scipy.special as special

NUM_PHASES = 4
G_MIN = 15.0
G_MAX = 60.0

ARRIVAL_RATES = np.array([0.25, 0.15, 0.30, 0.10])
INITIAL_QUEUES = np.array([12, 5, 20, 4])
SATURATION_FLOW = 0.5

def objective_function(green_times):
    """
    Evaluates a traffic signal timing plan.
    Returns a combined fitness score of average waiting time and residual queues.
    (Goal: Minimize this value)
    """
    w1, w2 = 0.6, 0.4

    total_waiting_time = 0
    remaining_queues = 0

    for i in range(NUM_PHASES):
        g = green_times[i]
        arrival = ARRIVAL_RATES[i]
        q_init = INITIAL_QUEUES[i]

        discharged = min(q_init + (arrival * g), SATURATION_FLOW * g)
        q_end = max(0.0, q_init + (arrival * g) - discharged)

        avg_queue_during_phase = (q_init + q_end) / 2.0
        waiting_time = avg_queue_during_phase * g

        total_waiting_time += waiting_time
        remaining_queues += q_end

    fitness = (w1 * total_waiting_time) + (w2 * remaining_queues)
    return fitness

def levy_flight(dim, beta=1.5):
    """Generates step sizes distributed according to Levy Flight."""
    sigma = (special.gamma(1 + beta) * np.sin(np.pi * beta / 2) /
             (special.gamma((1 + beta) / 2) * beta * 2**((beta - 1) / 2)))**(1 / beta)

    u = np.random.normal(0, sigma, size=dim)
    v = np.random.normal(0, 1, size=dim)

    step = u / (np.abs(v)**(1 / beta))
    return step

def cuckoo_search_traffic(n=25, pa=0.25, max_iter=100):

    nests = np.random.uniform(G_MIN, G_MAX, (n, NUM_PHASES))
    fitness = np.array([objective_function(nest) for nest in nests])

    best_idx = np.argmin(fitness)
    best_nest = nests[best_idx].copy()
    best_fitness = fitness[best_idx]

    alpha = 0.1

    for iteration in range(max_iter):

        for i in range(n):

            step = levy_flight(NUM_PHASES)
            new_nest = nests[i] + alpha * step * (nests[i] - best_nest)

            new_nest = np.clip(new_nest, G_MIN, G_MAX)
            new_fitness = objective_function(new_nest)

            j = np.random.randint(0, n)
            if new_fitness < fitness[j]:
                nests[j] = new_nest
                fitness[j] = new_fitness

        worst_indices = np.argsort(fitness)[-int(pa * n):]

        for idx in worst_indices:
            nests[idx] = np.random.uniform(G_MIN, G_MAX, NUM_PHASES)
            fitness[idx] = objective_function(nests[idx])

        current_best_idx = np.argmin(fitness)
        if fitness[current_best_idx] < best_fitness:
            best_fitness = fitness[current_best_idx]
            best_nest = nests[current_best_idx].copy()

        if (iteration + 1) % 20 == 0 or iteration == 0:
            print(f"Iteration {iteration+1:03d} | Current Minimum Congestion Index: {best_fitness:.4f}")

    return best_nest, best_fitness

if __name__ == "__main__":
    print("Initializing Cuckoo Search Optimization for Peak Hour Traffic...\n")

    optimal_timing, optimal_score = cuckoo_search_traffic(n=30, pa=0.2, max_iter=100)

    print("\n==============================================")
    print("🎯 OPTIMIZATION RESULT COMPLETED")
    print("==============================================")
    print(f"Optimized Phase Green Timings (seconds):")
    for phase, sec in enumerate(optimal_timing, start=1):
        print(f"  • Phase {phase}: {sec:.2f} seconds")
    print(f"Minimized Congestion Metric Score: {optimal_score:.4f}")


Initializing Cuckoo Search Optimization for Peak Hour Traffic...

Iteration 001 | Current Minimum Congestion Index: 366.1345
Iteration 020 | Current Minimum Congestion Index: 353.6947
Iteration 040 | Current Minimum Congestion Index: 353.6947
Iteration 060 | Current Minimum Congestion Index: 353.6947
Iteration 080 | Current Minimum Congestion Index: 353.6947
Iteration 100 | Current Minimum Congestion Index: 353.6947

🎯 OPTIMIZATION RESULT COMPLETED
Optimized Phase Green Timings (seconds):
  • Phase 1: 17.12 seconds
  • Phase 2: 15.81 seconds
  • Phase 3: 17.44 seconds
  • Phase 4: 23.30 seconds
Minimized Congestion Metric Score: 353.6947
